In [0]:
from pyspark.sql.functions import *

In [0]:
dbutils.widgets.text("landing_timestamp","")
dbutils.widgets.text("source_table_name","")
dbutils.widgets.text("merge_keys","")
dbutils.widgets.text("load_type","")


landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_table_name = dbutils.widgets.get("source_table_name")
load_type = dbutils.widgets.get("load_type")
merge_keys = dbutils.widgets.get("merge_keys")

In [0]:
bronze_df = spark.read.table(f"caresyncsydatabricks.bronze.{source_table_name}")\
                          .filter(col("landing_timestamp") == lit(landing_timestamp))

In [0]:
updated_df = bronze_df.withColumn("insert_timestamp",current_timestamp())\
                       .withColumn("updated_timestamp",current_timestamp())

In [0]:
target_table = f"caresyncsydatabricks.silver.{source_table_name}"
normalized_load_type = (load_type or 'FULL').strip().upper()

if normalized_load_type == "MERGE":
    merge_key_list = [key.strip() for key in merge_keys.split(",") if key.strip()]
    if not merge_key_list:
        raise ValueError("Keys must be available for merging")
    
    missing_keys = [key for key in merge_key_list if key not in bronze_df.columns]
    if missing_keys:
        raise ValueError("Keys are not available in bronze df")

if normalized_load_type  == "FULL":
    updated_df.write.mode("overwrite")\
            .saveAsTable(target_table)
elif normalized_load_type == "APPEND":
    updated_df.write.mode("append")\
            .saveAsTable(target_table)
elif normalized_load_type == "MERGE":
    if not spark.catalog.tableExists(target_table):
        updated_df.write.mode("overwrite")\
            .saveAsTable(target_table)
    else:
        source_view ="silver_merge_view"

        updated_df.createOrReplaceTempView(source_view)

        merge_condition = "AND".join([f"target.{key} = source.{key}" for key in merge_key_list])

        update_assign = ",".join([f"target.{keys} = source.{keys}" for keys in updated_df.columns if keys != "insert_timestamp"])

        insert_column = ",".join([keys for keys in updated_df.columns])

        insert_assign = ",".join([f"source.{keys}" for keys in updated_df.columns])


        spark.sql(f"MERGE INTO {target_table} as target \
                           Using {source_view} As source  \
                           ON {merge_condition} \
                           when matched then \
                           update set  \
                           {update_assign}  \
                            when not matched then \
                            insert ({insert_column}) \
                             values ({insert_assign})" )
        
else:
    raise ValueError("Not found load_type")





In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))